# NEMESIS — Fleet + Circuit Breaker (hosted notebook = the VM)

This notebook is the **remote VM**, not the local box. Run it on Colab / Kaggle / Binder
and the fleet executes on *that* machine's CPU/GPU — this box stays idle.

**Facts that matter (measured, not assumed):**
- A hosted notebook kernel is **server-side Python on a CUDA VM** (Colab T4/A100, Kaggle P100/T4).
  That is a **CUDA** GPU, **not WebGPU**. WebGPU is a *browser* API and does not exist in a kernel.
- The "fleet" here is the **orchestration + circuit breaker + guarded parallel pool**. It runs any
  callable. To make the workers call an LLM you must supply a model key for the VM (the local box's
  opencode gateway is not reachable from here).
- **OOM guard**: concurrency is capped against real `MemAvailable` and each worker's budget.

## 1. Environment — what VM did we land on?

In [ ]:
import os, sys, platform, subprocess
def sh(c):
    try: return subprocess.run(c, shell=True, capture_output=True, text=True, timeout=25).stdout.strip()
    except Exception as e: return f"err {e}"
print("python   ", sys.version.split()[0])
print("platform ", platform.platform())
print("cpu      ", os.cpu_count())
try:
    import psutil; vm = psutil.virtual_memory()
    print("ram      ", f"{vm.total/2**30:.1f} GiB total / {vm.available/2**30:.1f} GiB avail")
except Exception as e: print("psutil   ", e)
print("nvidia   ", sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader") or "(no GPU)")
try:
    import torch
    print("torch    ", torch.__version__, "| cuda:", torch.cuda.is_available(),
          "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"))
except Exception:
    print("torch     not installed (fine for CPU orchestration)")

## 2. Circuit breaker — faithful mirror of `covenant.js`

`THRESHOLD=3` failures open the circuit; `COOLDOWN=30s` -> half-open probes; a success closes it.

In [ ]:
import time, threading
class CircuitBreaker:
    def __init__(self, threshold=3, cooldown=30.0):
        self.threshold, self.cooldown = threshold, cooldown
        self.fails, self.state, self.opened_at = 0, "closed", None
        self._lk = threading.Lock()
    def allow(self):
        with self._lk:
            if self.state == "open" and self.opened_at and time.time() - self.opened_at >= self.cooldown:
                self.state = "half"
            return self.state != "open"
    def success(self):
        with self._lk:
            if self.state == "half":
                self.state, self.fails, self.opened_at = "closed", 0, None
    def failure(self):
        with self._lk:
            self.fails += 1
            if self.fails >= self.threshold and self.state != "open":
                self.state, self.opened_at = "open", time.time()
    def snapshot(self):
        with self._lk: return {"state": self.state, "fails": self.fails}
breaker = CircuitBreaker()
breaker.snapshot()

## 3. OOM guard — cap concurrency against real free RAM

In [ ]:
import psutil
def safe_workers(per_worker_mb=700, floor_mb=900, cap=None):
    avail = psutil.virtual_memory().available / 2**20
    if avail <= floor_mb: return 0, avail
    n = int((avail - floor_mb) // per_worker_mb)
    if cap: n = min(n, cap)
    return max(1, n), avail

sw, avail = safe_workers(per_worker_mb=256)
print(f"MemAvailable={avail:.0f} MB -> safe workers (256MB each, 900MB floor) = {sw}")

## 4. Guarded parallel pool + breaker

Every task is retried across the breaker; when the circuit opens, dispatch fails fast.
Replace `task()` with real work (or an LLM call using a key for this VM).

In [ ]:
import concurrent.futures as cf, time, random, os
def task(i, flaky=0.15):
    if random.random() < flaky: raise RuntimeError(f"flaky worker {i}")
    s = 0
    for k in range(200_000): s += k*k      # real CPU-bound work
    return i, s

def run_fleet(n=24, per_worker_mb=256, max_attempts=3, cap=None):
    workers, avail = safe_workers(per_worker_mb=per_worker_mb, cap=cap)
    print(f"deploy {n} tasks | safe_workers={workers} | MemAvail={avail:.0f}MB | breaker={breaker.snapshot()}")
    done, failed, retried = 0, 0, 0
    pending = list(range(n))
    with cf.ThreadPoolExecutor(max_workers=max(1, workers)) as ex:
        futs = {ex.submit(task, i): i for i in pending}
        for fut in cf.as_completed(futs):
            if not breaker.allow():
                failed += 1; continue
            i = futs[fut]
            try:
                fut.result(); breaker.success(); done += 1
            except Exception as e:
                breaker.failure()
                # retry while circuit closed
                tries = 1
                while tries < max_attempts and breaker.allow():
                    try:
                        task(i); breaker.success(); done += 1; retried += 1; break
                    except Exception:
                        breaker.failure(); tries += 1
                else:
                    failed += 1
    print(f"done={done} failed={failed} retried={retried} breaker={breaker.snapshot()}")
    return done, failed

run_fleet(n=24)

## 5. WebGPU vs the notebook GPU — the distinction

| | where it runs | what you get | usable in a notebook kernel? |
|---|---|---|---|
| **CUDA** | server-side GPU on the VM | `torch`, cuPy, numba | **yes** — this is the notebook GPU |
| **WebGPU** | **browser** (client JS) | WGSL compute in a Web Worker | **no** — no `navigator.gpu` in a kernel |

"Offload memory to WebGPU" applies to the **browser** client (`nemesis-browser-llm.html`), not to a
hosted notebook. In a notebook, the offload target is **CUDA**.

## 6. Adding real LLM workers

The local box's `opencode` gateway is **not reachable** from this VM. To run model workers here,
add a key (e.g. an HF token or an API key) as a Colab secret, then wrap the call in `task()`.

## 7. Wire the notebook to the message board and the gateway

The local box exposes a **narrow, key-gated bridge** (not the raw gateway — that has no auth). The
bridge holds the GitHub token **server-side**, so this notebook never carries it. Public endpoint:

| route | method | what |
|---|---|---|
| `/state` | GET | governance + covenant + fleet + mem + identity snapshot |
| `/board/recent?n=` | GET | last N records of `atrium/board/board.jsonl` |
| `/board/append` | POST | append one record (`{record:{…}}`), returns `{commit, refMatches}` |
| `/rpa/run` | POST | ask the box to run an RPA unit (`{id}`) |

The URL/key are supplied at runtime (env or prompt) — **never committed**. A quick tunnel is
ephemeral: its `*.trycloudflare.com` host changes each time and dies with the tunnel process.

In [ ]:
import os, json, time, urllib.request, urllib.error, getpass

GATEWAY_URL = os.environ.get("NEMESIS_GATEWAY_URL", "").strip() or \
    getpass.getpass("Nemesis gateway URL (https://…): ").strip()
GATEWAY_KEY = os.environ.get("NEMESIS_BRIDGE_KEY", "").strip() or \
    getpass.getpass("Nemesis bridge key: ")

def _req(path, method="GET", payload=None):
    url = GATEWAY_URL.rstrip("/") + path
    data = json.dumps(payload).encode() if payload is not None else None
    req = urllib.request.Request(url, data=data, method=method,
        headers={"x-nemesis-key": GATEWAY_KEY, "content-type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=40) as r:
            return r.status, json.loads(r.read().decode())
    except urllib.error.HTTPError as e:
        return e.code, {"error": e.read().decode()[:300]}
    except Exception as e:
        return 0, {"error": str(e)}

def gateway_state():        return _req("/state")
def board_recent(n=20):     return _req(f"/board/recent?n={n}")
def board_append(record):   return _req("/board/append", "POST", {"record": record})
def rpa_run(unit_id):       return _req("/rpa/run", "POST", {"id": unit_id})

print("gateway:", GATEWAY_URL or "(unset)")
print("state  :", json.dumps(gateway_state()[1], indent=1)[:700])

### 7.1 Round trip — read the board, compute, append a result

This proves the notebook is a first-class **seat**: it reads the live board and writes back
(read-back verified with `refMatches`).

In [ ]:
# announce this seat
announce = {"type": "NOTEBOOK_SEAT", "agent": "nemesis-notebook", "surface": "binder",
            "ts": int(time.time()), "verdict": "ALIVE",
            "note": "hosted notebook wired to board + gateway"}
print("append:", board_append(announce))

# read back the tail + summarize
st, b = board_recent(50)
rows = b.get("records", [])
kinds = {}
for r in rows:
    kinds[r.get("type", "?")] = kinds.get(r.get("type", "?"), 0) + 1
result = {"scanned": len(rows), "board_total": b.get("total"), "kinds": kinds}
print("result :", result)
print("write  :", board_append({"type": "NOTEBOOK_RESULT", "agent": "nemesis-notebook",
       "ts": int(time.time()), "verdict": "PASS", "result": result}))